**VISUAL SERIES**

In [ ]:
import digitalhub as dh

NOME_PROGETTO = "floods"
project = dh.get_project(NOME_PROGETTO)
print(f"Progetto: {project.name}")

**SETUP PARAMETERS**

In [ ]:
# Parametri Job   
job_name = "anom_v2_v2"                                
dataset = "Floods_Anomalies_v2_crop_norm" 
test_sar = True
test_opt = True                                       
#handler = pretrain_encoders   
# Floods_Test_crop_norm
# Floods_Anomalies_100_crop_norm                                       

parametri = {     
    "patch_size": 256, 
    "n_images1": 4, "n_channels1": 2,                      
    "n_images2": 4, "n_channels2": 10,  
    "output_dim": 16,                              
    "mamba": False, 
    "dataset": dataset,
    "test_sar": test_sar,                                 
    "test_opt": test_opt,
    "weights_sar": "autoencoder_2D_SAR_weights_convLstm_v1",
    "weights_opt": "autoencoder_2D_OPT_weights_convLstm_v1",
    "job_name": job_name,
    "ltae": False,
    "monodimensional": False,
    "n_samples": 10,    
    "sar_channel": 0,     
    "recon_channels": [2,1,0]
}                                                              

print(f"PARAMETRI: {parametri}")

# volume -> circa 400 GB dataset Standard
volumi = [
    {
        "volume_type": "ephemeral",
        "name": "volume-spazio-dati",
        "mount_path": "/data",      
        "spec": {"size": "200Gi"}   
    }
]

**BUILD ENVIRONMENT**

In [ ]:
test_train_func = project.new_function(
    name= f'visual_series_{job_name}',
    kind="python",
    python_version="PYTHON3_10",
    code_src="../src/", 
    handler="utils:visual_series", 
    base_image="pytorch/pytorch:2.1.2-cuda11.8-cudnn8-runtime",
    requirements=["pandas==2.3.3", "numpy==1.26.4", "rasterio==1.4.4", "tqdm==4.70.0", "tifffile==2024.8.30", "torch==2.1.2", "matplotlib==3.10.9", "digitalhub==0.15.11", "digitalhub-runtime-python==0.15.2", "zarr==2.18.0", "numcodecs==0.13.1", "xarray==2025.6.1"]
)

# .run("build") -> scarica immagine, installa requirements, copia intera cartella code_src, crea immagine docker

build = test_train_func.run("build", wait=True)
print(f"BUILD: {build.status.state}")

**TRAINING**

In [ ]:
# action job = avvia container, esegue script, libera risorse

run_test_encoders = test_train_func.run(
    action="job", 
    parameters=parametri, 
    volumes=volumi, 
    profile="1xv100-shared",                                       # 1x = 1 gpu
    # local_execution= True,                                
    wait=True
)

print(f"Run test_encoders avviato: {run_test_encoders.id}")
print(run_test_encoders.status.state)
print(run_test_encoders.status.message)